# F-S-0-RES: Reconstruction Error vs. Angular Separation

Single-frequency, noiseless angular-resolution check. This measures reconstruction/conditioning, not support detection.


### 0. Imports


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.fft import fftfreq


def find_repo_root(start=Path.cwd()):
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "src" / "cs_priors").exists():
            return path
    raise RuntimeError("Could not find repository root")


REPO_ROOT = find_repo_root()
sys.path[:0] = [
    str(REPO_ROOT / "src"),
    str(REPO_ROOT / "notebooks" / "benchmarks_v2" / "functions"),
]

from cs_priors.plotting.plot_complex import plot_matrices
from cs_priors.plotting.plot_geometry import plot_sim_geometry
from cs_priors.plotting.plot_signals import plot_recovery
from figures import save_pdf
from single_frequency_benchmarks import (
    METHOD_LABELS,
    METHOD_ORDER,
    make_simulation,
    plot_metric,
    relabel_legend,
    run_angle_separation_benchmark,
    solve_methods,
)
from single_frequency_config import (
    LASSO_ALPHA,
    LASSO_MAX_ITER,
    PHASE_SEEDS,
    base_sim_kwargs,
)


### 1. Parameters


In [ ]:
TAG = "F-S-0-RES"
FIGURE_DIR = REPO_ROOT / "results" / "benchmarks" / "v2" / TAG
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SEPARATIONS_DEG = np.array([
    1e-9, 3e-9,
    1e-8, 3e-8,
    1e-7, 3e-7,
    1e-6, 3e-6,
    1e-5, 3e-5,
    1e-4, 3e-4,
    1e-3, 3e-3,
    1e-2, 3e-2,
    1e-1, 3e-1,
    1e0, 3e0,
    1e1, 3e1,
])

NUM_MICS = 2
NUM_SOURCES = NUM_MICS
NUM_ACTIVE = NUM_SOURCES
METHODS_TO_PLOT = METHOD_ORDER

SIM_KWARGS = base_sim_kwargs(
    num_mics=NUM_MICS,
    num_sources=NUM_SOURCES,
    num_active=NUM_ACTIVE,
)

SIM_KWARGS["mic_angle_span"] = np.deg2rad(360)


### 2. Sanity check


In [ ]:
CHECK_SEPARATION_DEG = SEPARATIONS_DEG[-1]
# GEOMETRY_VIEW_LIMITS = ((-0.03, 0.35), (-0.03, 0.18))
# GEOMETRY_GRID_STEP = 0.02

def add_inward_source_arrows(ax, sim, radius_factor=1.01, frac=0.55, offset_deg=4):
    active = list(sim.active_indices)
    angles = np.array([sim.sources[i].angle for i in active])
    a0, a1 = angles.min(), angles.max()
    amid = 0.5 * (a0 + a1)
    radius = sim.sources[active[0]].distance * radius_factor
    offset = np.deg2rad(offset_deg)

    for angle, rad in [(a0, 0.10), (a1, -0.10)]:
        side = np.sign(amid - angle)
        target = angle + frac * (amid - angle) - side * offset

        ax.annotate(
            "",
            xy=(radius * np.cos(target), radius * np.sin(target)),
            xytext=(radius * np.cos(angle), radius * np.sin(angle)),
            arrowprops=dict(
                arrowstyle="-|>",
                connectionstyle=f"arc3,rad={rad}",
                lw=1.8,
                color="green",
                mutation_scale=12,
            ),
            zorder=20,
        )


def expand_single_frequency_spectrum(X_one, sim):
    N = int(sim.sampling_rate * sim.duration)
    freqs = fftfreq(N, d=1 / sim.sampling_rate)
    X_full = np.zeros((X_one.shape[0], N), dtype=complex)

    for j, freq in enumerate(sim.freqs):
        k = np.argmin(np.abs(freqs - freq))
        if not np.isclose(freqs[k], freq):
            raise ValueError(f"Could not find FFT bin for {freq} Hz")

        X_full[:, k] = X_one[:, j]
        X_full[:, (-k) % N] = np.conj(X_one[:, j])

    return X_full


sim = make_simulation(CHECK_SEPARATION_DEG, PHASE_SEEDS[0], **SIM_KWARGS)
solutions = solve_methods(sim, lasso_alpha=LASSO_ALPHA, lasso_max_iter=LASSO_MAX_ITER)


fig, ax = plot_sim_geometry(
    sim,
    dpi=90,
    pad_factor=0.1,
    show=False,
    unit="cm",
    # view_limits=GEOMETRY_VIEW_LIMITS,
    # grid_step=GEOMETRY_GRID_STEP,
)
add_inward_source_arrows(ax, sim, offset_deg=0, frac=1, radius_factor=1.03)
save_pdf(fig, FIGURE_DIR / "simulation_geometry_angle_separation_check.pdf")
plt.show()

plot_matrices(
    [sim.X, *solutions.values()],
    [r"$\boldsymbol{x}$ true", *[METHOD_LABELS.get(k, k) for k in solutions.keys()]],
    dpi=60,
    show_values=True,
)


### 3. Generating results


In [ ]:
results_df = run_angle_separation_benchmark(
    SEPARATIONS_DEG,
    PHASE_SEEDS,
    sim_kwargs=SIM_KWARGS,
    lasso_alpha=LASSO_ALPHA,
    lasso_max_iter=LASSO_MAX_ITER,
)

results_df["method"] = pd.Categorical(
    results_df["method"], categories=METHODS_TO_PLOT, ordered=True
)
results_df = results_df.sort_values(
    ["separation_deg", "phase_seed", "method"]
).reset_index(drop=True)
results_df.head()


### 4. Plotting results


In [ ]:
fig, ax, summary_df = plot_metric(
    results_df,
    method_order=METHODS_TO_PLOT,
    yscale="linear",
    xscale="log",
)

relabel_legend(ax, METHOD_LABELS)

from matplotlib.ticker import LogFormatterSciNotation
ax.set_xticks(SEPARATIONS_DEG)
ax.xaxis.set_major_formatter(LogFormatterSciNotation())
ax.tick_params(axis="x", labelrotation=45)

save_pdf(fig, FIGURE_DIR / "relative_complex_error_vs_angle_separation.pdf")
plt.show()

summary_df.head()

